In [ ]:
# STEP 1: Install Dependencies

!pip install langchain langchain-chroma chromadb sentence-transformers transformers accelerate streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 

In [ ]:
!pip install langchain-huggingface

In [ ]:
# STEP 2: Login to Hugging Face

import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
login(token=hf_token)

In [ ]:
# STEP 3: Import Libraries

from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from transformers import pipeline
import re

In [ ]:
# STEP 4: Load Multiple Meeting Sample Transcript

transcripts = {
    "team_sync_2026-03-20": """
    [2026-03-20] John: We should launch the product next Friday.
    [2026-03-20] Sarah: I'll prepare the marketing plan.
    [2026-03-20] Mike: The backend APIs are almost ready.
    [2026-03-20] Lisa: We need final approval from management.
    """,

    "team_sync_2026-03-21": """
    [2026-03-21] Mike: The API is still unstable.
    [2026-03-21] John: Let's fix the API before launch.
    [2026-03-21] Sarah: Marketing content draft is completed.
    [2026-03-21] Lisa: We may need to delay the launch.
    """,

    "team_sync_2026-03-22": """
    [2026-03-22] John: We decided to postpone the launch by one week.
    [2026-03-22] Mike: I'll work on stabilizing the API.
    [2026-03-22] Sarah: I'll update the campaign timeline.
    [2026-03-22] Rachel: Client demo is scheduled for Thursday.
    """,

    "team_sync_2026-03-23": """
    [2026-03-23] Mike: API performance has improved.
    [2026-03-23] John: Good, let's continue testing.
    [2026-03-23] Lisa: QA team found a few bugs.
    [2026-03-23] Rachel: I'll coordinate with QA for fixes.
    """,

    "team_sync_2026-03-24": """
    [2026-03-24] John: We decided to fix all critical bugs before demo.
    [2026-03-24] Mike: I'll prioritize critical issues.
    [2026-03-24] Sarah: Campaign launch date is tentative.
    [2026-03-24] Lisa: We should inform stakeholders about delay.
    """,

    "team_sync_2026-03-25": """
    [2026-03-25] Rachel: Client demo went well.
    [2026-03-25] John: Great job team.
    [2026-03-25] Mike: Some minor bugs still exist.
    [2026-03-25] Sarah: I'll finalize marketing assets.
    """,

    "team_sync_2026-03-26": """
    [2026-03-26] John: We decided to finalize the launch date as April 5.
    [2026-03-26] Lisa: I'll send confirmation to stakeholders.
    [2026-03-26] Mike: API is stable now.
    [2026-03-26] Rachel: Documentation needs updates.
    """,

    "team_sync_2026-03-27": """
    [2026-03-27] Sarah: Marketing campaign is ready.
    [2026-03-27] John: Good, let's proceed with launch preparation.
    [2026-03-27] Mike: Monitoring tools are set up.
    [2026-03-27] Lisa: We should prepare a backup plan.
    """,

    "team_sync_2026-03-28": """
    [2026-03-28] John: We decided to keep a rollback plan ready.
    [2026-03-28] Mike: I'll configure rollback scripts.
    [2026-03-28] Rachel: Support team is briefed.
    [2026-03-28] Sarah: Social media posts are scheduled.
    """,

    "team_sync_2026-03-29": """
    [2026-03-29] Lisa: We decided to buy the product on next Monday.
    [2026-03-29] John: Ask the administrator to fix Mike's PC.
    [2026-03-29] Mike: My system is still facing issues.
    [2026-03-29] Sarah: I'll check with IT support.
    """,

    "team_sync_2026-03-30": """
    [2026-03-30] Rachel: The product for X company didn't get shipped yesterday.
    [2026-03-30] John: We should do meetings once every week.
    [2026-03-30] Lisa: Let's track shipment delays more closely.
    [2026-03-30] Mike: I'll investigate the shipping issue.
    """
}

In [ ]:
# STEP 5: Prepare Documents & Chunking with Metadata

docs = []

def extract_date(text):
  match = re.search(r"\[(.*?)]" , text)
  return match.group(1) if match else None

seen = set()

for meeting_id , text in transcripts.items():
  chunks = [line.strip() for line in text.split("\n") if line.strip()]



  for chunk in chunks:
    if chunk not in seen:
      seen.add(chunk)
      docs.append(Document(
          page_content = chunk ,
          metadata ={

              "date" : extract_date(chunk) ,
              "meeting_id" : meeting_id
        }
  ))


for d in docs:
  print(d.page_content , d.metadata)


[2026-03-20] John: We should launch the product next Friday. {'date': '2026-03-20', 'meeting_id': 'team_sync_2026-03-20'}
[2026-03-20] Sarah: I'll prepare the marketing plan. {'date': '2026-03-20', 'meeting_id': 'team_sync_2026-03-20'}
[2026-03-20] Mike: The backend APIs are almost ready. {'date': '2026-03-20', 'meeting_id': 'team_sync_2026-03-20'}
[2026-03-20] Lisa: We need final approval from management. {'date': '2026-03-20', 'meeting_id': 'team_sync_2026-03-20'}
[2026-03-21] Mike: The API is still unstable. {'date': '2026-03-21', 'meeting_id': 'team_sync_2026-03-21'}
[2026-03-21] John: Let's fix the API before launch. {'date': '2026-03-21', 'meeting_id': 'team_sync_2026-03-21'}
[2026-03-21] Sarah: Marketing content draft is completed. {'date': '2026-03-21', 'meeting_id': 'team_sync_2026-03-21'}
[2026-03-21] Lisa: We may need to delay the launch. {'date': '2026-03-21', 'meeting_id': 'team_sync_2026-03-21'}
[2026-03-22] John: We decided to postpone the launch by one week. {'date': '2

In [ ]:
# STEP 6: Create Embeddings + Chroma DB

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = Chroma.from_documents(docs , embedding)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
## Phase 2 - Add Time-Aware + Structured Retrieval UI

In [ ]:
# STEP 7: Add SMART QUERY UNDERSTANDING

import re
from datetime import datetime, timedelta

def parse_query(query):
    query_lower = query.lower()
    time_filter = None
    today = datetime.today().date()

    range_match = re.search(r"(\d{4}-\d{2}-\d{2}).*?(\d{4}-\d{2}-\d{2})", query_lower)
    if range_match:
        return {"type": "date_range", "start": range_match.group(1), "end": range_match.group(2)}

    exact_match = re.search(r"\d{4}-\d{2}-\d{2}", query_lower)
    if exact_match:
        return {"type": "exact_date", "date": exact_match.group(0)}

    natural_match = re.search(
        r"(\d{1,2})\s+(january|february|march|april|may|june|july|august|september|october|november|december)(?:\s+(\d{4}))?",
        query_lower
    ) or re.search(
        r"(january|february|march|april|may|june|july|august|september|october|november|december)\s+(\d{1,2})(?:\s+(\d{4}))?",
        query_lower
    )

    if natural_match:
        groups = natural_match.groups()
        try:
            if groups[0].isdigit():
                day, month_str, year = groups[0], groups[1], groups[2] or str(today.year)
            else:
                month_str, day, year = groups[0], groups[1], groups[2] or str(today.year)
            parsed = datetime.strptime(f"{day} {month_str} {year}", "%d %B %Y").date()
            return {"type": "exact_date", "date": str(parsed)}
        except ValueError:
            pass

    if "today" in query_lower:
        return {"type": "today", "date": str(today)}

    if "yesterday" in query_lower:
        return {"type": "yesterday", "date": str(today - timedelta(days=1))}

    if "last week" in query_lower:
        return {"type": "last_week", "start": str(today - timedelta(days=7)), "end": str(today)}

    return time_filter

In [ ]:
# STEP 8: Apply FILTERS before retrieval


from datetime import datetime, timedelta

def filter_docs(docs, time_filter=None):
    filtered = docs





    if time_filter:


        if time_filter["type"] == "last_week":
            start = datetime.strptime(time_filter["start"], "%Y-%m-%d")
            end = datetime.strptime(time_filter["end"], "%Y-%m-%d")

            filtered = [
                d for d in filtered
                if d.metadata.get("date")
                and start <= datetime.strptime(d.metadata["date"], "%Y-%m-%d") <= end
            ]


        elif time_filter["type"] == "exact_date":
            target = time_filter["date"]

            filtered = [
                d for d in filtered
                if d.metadata.get("date") == target
            ]

        elif time_filter["type"] in ["today", "yesterday"]:
          target = time_filter["date"]

          filtered = [
              d for d in filtered
              if d.metadata.get("date") == target
            ]


        elif time_filter["type"] == "date_range":
            start = datetime.strptime(time_filter["start"], "%Y-%m-%d")
            end = datetime.strptime(time_filter["end"], "%Y-%m-%d")

            filtered = [
                d for d in filtered
                if d.metadata.get("date")
                and start <= datetime.strptime(d.metadata["date"], "%Y-%m-%d") <= end
            ]

    return filtered

In [ ]:
# STEP 9: Retrieval

from chromadb import EphemeralClient

def hybrid_search(query, db, filtered_docs, k=3):
    if not filtered_docs:
        return []

    client = EphemeralClient()
    temp_db = Chroma.from_documents(
        filtered_docs,
        embedding,
        client=client,
        collection_name="temp_search"
    )

    semantic_docs = temp_db.similarity_search(query, k=k)

    seen = set()
    combined = []
    for d in semantic_docs:
        if d.page_content not in seen:
            combined.append(d)
            seen.add(d.page_content)

    return combined[:k]

In [ ]:
def build_prompt(query, context):
    query_lower = query.lower()

    if "decision" in query_lower:
        instruction = "List only the decisions made in these meeting notes."
    elif "action" in query_lower or "task" in query_lower:
        instruction = "List only the action items and tasks assigned in these meeting notes."
    elif "discuss" in query_lower or "point" in query_lower or "summary" in query_lower or "main" in query_lower:
        instruction = "Summarize the key discussion points from these meeting notes."
    else:
        instruction = "Answer the question based on these meeting notes."

    return f"""{instruction}

Context:
{context}

Answer:"""

In [ ]:
# STEP 10: RAG pipeline

from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "google/flan-t5-xl"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

def rag_pipeline(query):
    time_filter = parse_query(query)
    filtered_docs = filter_docs(docs, time_filter)

    if not filtered_docs:
        return "I don't know", []

    results = hybrid_search(query, db, filtered_docs)

    if not results:
        return "I don't know", []

    seen = set()
    unique_results = []
    for r in results:
        if r.page_content not in seen:
            unique_results.append(r)
            seen.add(r.page_content)

    results = unique_results
    context = "\n".join([r.page_content for r in results])
    query_clean = query.strip()

    prompt = f"""You are a meeting assistant.
Use the context to answer the question.
If no answer exists, reply exactly: "No records found."

Context:
{context}

Question:
{query_clean}

Summarize the key points as a short list:"""



    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer, results

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
# STEP 11: SHOW RETRIEVAL

query = "tell the main discussions from last week . tell everything from 23rd to 27"

answer , results = rag_pipeline(query)

print("ANSWER :\n" , answer)
print("\nRETRIEVED CHUNKS:\n")

for r in results:
  print(r.page_content)

ANSWER :
 No records found

RETRIEVED CHUNKS:

[2026-03-30] John: We should do meetings once every week.
[2026-03-28] Sarah: Social media posts are scheduled.
[2026-03-26] Lisa: I'll send confirmation to stakeholders.


In [ ]:
# STEP 14: UI (Streamlit) & Timeline/Chart

import streamlit as st
from datetime import datetime
import pandas as pd

st.title("Meeting Intelligence RAG")

query = st.text_input("Ask your question")
tag_filter = st.selectbox("Filter by type:", ["all", "decision", "action_item", "discussion"])
date_filter = st.date_input("Filter by date (optional)")

if query:
    filtered_docs = docs

    if tag_filter != "all":
        filtered_docs = [d for d in filtered_docs if d.metadata["tag"] == tag_filter]

    if date_filter:
        filtered_docs = [
            d for d in filtered_docs
            if d.metadata["date"] and datetime.strptime(d.metadata["date"], "%Y-%m-%d") == date_filter
        ]

    answer, result = rag_pipeline(query)

    st.subheader("Answer")
    st.write(answer)

    st.subheader("Retrieved Context")
    for r in result:
        st.write(r.page_content)
        st.write(r.metadata)

    st.subheader("Chunks contributing to answer")
    for i, r in enumerate(result):
        st.write(f"Chunk {i+1}")
        st.write(r.page_content)
        st.write(r.metadata)

    timeline = pd.DataFrame([
        {"date": d.metadata["date"], "tag": d.metadata["tag"], "content": d.page_content}
        for d in docs if d.metadata.get("date")
    ])

    timeline = timeline.sort_values("date")

    st.subheader("Timeline View")
    st.line_chart(pd.crosstab(timeline["date"], timeline["tag"]))

2026-04-01 10:11:52.640 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 10:11:52.859 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-04-01 10:11:52.860 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 10:11:52.862 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 10:11:52.863 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 10:11:52.865 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 10:11:52.867 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 10:11:52.869 Thread 'MainThread': mi

In [ ]:
# STEP 15: Add “Action Items” View , Timeline View , Multi-meeting support
'''
tasks = [d for d in docs if d.metadata["tag"] == "action_item"]

sorted_docs = sorted(docs , key = lambda x : x.metadata["date"])

for t in tasks:
  print("Date:" , t.metadata.get("date"))
  print("Meeting ID:" , t.metadata.get("meeting_id" , "N/A"))
  print("Content:" , t.page_content)
  print("---")

print("Timeline View:")
for d in sorted_docs:
  print(
      d.metadata.get("date"),
      d.metadata.get("meeting_id" , "N/A"),
      d.metadata["tag"],
      d.page_content
  )
  '''

'\ntasks = [d for d in docs if d.metadata["tag"] == "action_item"]\n\nsorted_docs = sorted(docs , key = lambda x : x.metadata["date"])\n\nfor t in tasks:\n  print("Date:" , t.metadata.get("date"))\n  print("Meeting ID:" , t.metadata.get("meeting_id" , "N/A"))\n  print("Content:" , t.page_content)\n  print("---")\n\nprint("Timeline View:")\nfor d in sorted_docs:\n  print(\n      d.metadata.get("date"),\n      d.metadata.get("meeting_id" , "N/A"),\n      d.metadata["tag"],\n      d.page_content\n  )\n  '